# PTX Puzzles

Twelve small CUDA kernels, each missing a piece of synchronization, usually between thread blocks. Each one gives the right answer on today's GPUs, and the PTX memory model still allows it to give the wrong one. Fix each kernel between its `// >>> edit` and `// <<< edit` markers, then run the grade cell under it. [syncsan](https://github.com/ShreshthRajan/syncsan) grades the fix, so a kernel that only happens to work on this GPU fails.

Pick a GPU runtime first (Runtime, then Change runtime type, then T4 GPU) and run the setup cell. Puzzle 8 needs an H100 or newer and is skipped on a T4.

## What you need to know

Blocks talk through global memory. One block writes its data and then sets a flag or bumps a counter, and the other spins until it sees the flag and then reads. The writer has to publish with a **release** and the reader has to wait with an **acquire**. Once the acquire reads the value the release wrote, the reader sees everything the writer wrote before the release. If either half is missing, the reader can see the flag and still read old data.

Plain loads and stores are weak. Relaxed, volatile, acquire, release and atomic accesses are strong. Two accesses to the same word race when one is a write, one is weak, and nothing orders them.

The tools are in [common.cuh](https://github.com/ShreshthRajan/ptx-puzzles/blob/main/puzzles/common.cuh):

- `st_relaxed`, `ld_relaxed` and `volatile` accesses are strong and order nothing else.
- `st_release(p, v)` is a flag store that publishes everything this thread has seen. `ld_acquire(p)` is a flag load: once it reads a published value, this thread also sees everything published with it.
- `atomicAdd`, `atomicSub`, `atomicCAS` and `atomicExch` are strong, and on their own they order nothing else.
- `__threadfence()` and `fence_acq_rel()` are fences. Before a strong store or an atomic, a fence makes it a release. After a strong load or an atomic, a fence makes it an acquire.
- `__syncthreads()` is a barrier for the block. `bar_sync(id, n)` and `bar_arrive(id, n)` are a barrier for n threads, where a thread's writes before its `bar_arrive` come before the reads after the matching `bar_sync`.
- `fence_proxy_async()` is needed in puzzle 8. It orders this thread's earlier shared-memory writes before the TMA unit reads them.

Scoring: a passing fix is scored by the barriers, fences, acquires and releases you added, against par, the count in the reference solution. Changing a plain access to an atomic costs nothing, so par can be 0.

In [ ]:
%%bash
# Setup, about a minute: install syncsan, build it for this GPU, fetch the puzzles.
pip -q install syncsan==0.10.5
syncsan build --arch $(nvidia-smi --query-gpu=compute_cap --format=csv,noheader | head -1 | tr -d .) > build.log 2>&1 && echo 'syncsan built' || tail -20 build.log
[ -d ptx-puzzles ] || git clone -q https://github.com/ShreshthRajan/ptx-puzzles
cd ptx-puzzles && python grade.py --help > /dev/null && echo 'puzzles ready'

## Puzzle 1: Message passing

In [ ]:
%%writefile ptx-puzzles/puzzles/01_message_passing.cu
// Puzzle 1: Message passing
//
// Block 1 writes 32 words, then sets a flag. Block 0 waits until it sees the flag, then copies the words.
//
// Make block 0 copy the words block 1 wrote. Par: 2.
//
// From: Hazy Research's Llama-1B megakernel (HazyResearch/Megakernels#16).
#include "common.cuh"

constexpr int N = 32;

__global__ void message_passing(unsigned* flag, unsigned* data, unsigned* out) {
  if (threadIdx.x != 0) return;
  if (blockIdx.x == 1) {                          // producer
    for (int i = 0; i < N; ++i) data[i] = i + 1;
    // >>> edit
    st_relaxed(flag, 1);
    // <<< edit
  } else {                                        // consumer
    // >>> edit
    while (ld_relaxed(flag) == 0) {}
    // <<< edit
    for (int i = 0; i < N; ++i) out[i] = data[i];
  }
}

int main() {
  unsigned* flag = device_zeros<unsigned>(1);
  unsigned* data = device_zeros<unsigned>(N);
  unsigned* out = device_zeros<unsigned>(N);
  message_passing<<<2, 32>>>(flag, data, out);
  finish_launch();
  unsigned got[N], want[N];
  host_copy(got, out, N);
  for (int i = 0; i < N; ++i) want[i] = i + 1;
  return check("out", got, want, N);
}


In [ ]:
!cd ptx-puzzles && python grade.py 1

## Puzzle 2: Waiting on a counter

In [ ]:
%%writefile ptx-puzzles/puzzles/02_counter_wait.cu
// Puzzle 2: Waiting on a counter
//
// Block 1's 128 threads each write one element of x, and the block syncs. Thread 0 then calls __threadfence() and
// increments a counter. In block 0, thread 0 spins on the counter with a volatile load, the block syncs, and every thread reads x.
//
// Make block 0's reads see block 1's writes. Par: 1.
//
// From: megakernel-zoo, Lucebox and AutoMegaKernel.
#include "common.cuh"

constexpr int T = 128;

__global__ void handoff(unsigned* ready, float* x, float* y) {
  const int t = threadIdx.x;
  if (blockIdx.x == 1) {                          // producer
    x[t] = 2.0f * t;
    __syncthreads();
    if (t == 0) { __threadfence(); atomicAdd(ready, 1u); }
  } else {                                        // consumer
    if (t == 0) {
      while (*(volatile unsigned*)ready == 0) {}
      // >>> edit
      // <<< edit
    }
    __syncthreads();
    y[t] = x[t] + 1.0f;
  }
}

int main() {
  unsigned* ready = device_zeros<unsigned>(1);
  float* x = device_zeros<float>(T);
  float* y = device_zeros<float>(T);
  handoff<<<2, T>>>(ready, x, y);
  finish_launch();
  float got[T], want[T];
  host_copy(got, y, T);
  for (int t = 0; t < T; ++t) want[t] = 2.0f * t + 1.0f;
  return check("y", got, want, T);
}


In [ ]:
!cd ptx-puzzles && python grade.py 2

## Puzzle 3: A triangular solve

In [ ]:
%%writefile ptx-puzzles/puzzles/03_triangular_solve.cu
// Puzzle 3: A triangular solve
//
// One step of a triangular solve. Four producer blocks each add their contribution into lsum with atomicAdd, then
// decrement a counter, fmod. The consumer block spins until fmod reaches 0, calls __threadfence(), and reads lsum.
//
// Make the consumer read the full sum. Par: 1.
//
// From: SuperLU_DIST's GPU triangular solve (xiaoyeli/superlu_dist#261, fixed upstream in 239280c).
#include "common.cuh"

constexpr int T = 128, P = 4;

__global__ void solve_step(int* fmod, float* lsum, float* x) {
  const int t = threadIdx.x;
  if (blockIdx.x > 0) {                           // producers 1..P
    atomicAdd(&lsum[t], (float)(blockIdx.x * t));
    __syncthreads();
    if (t == 0) {
      // >>> edit
      atomicSub(fmod, 1);
      // <<< edit
    }
  } else {                                        // consumer
    if (t == 0) { while (atomicAdd(fmod, 0) != 0) {} __threadfence(); }
    __syncthreads();
    x[t] = lsum[t] + 1.0f;
  }
}

int main() {
  const int deps = P;
  int* fmod = device_copy(&deps, 1);
  float* lsum = device_zeros<float>(T);
  float* x = device_zeros<float>(T);
  solve_step<<<1 + P, T>>>(fmod, lsum, x);
  finish_launch();
  float got[T], want[T];
  host_copy(got, x, T);
  for (int t = 0; t < T; ++t) want[t] = (float)(t * P * (P + 1) / 2) + 1.0f;
  return check("x", got, want, T);
}


In [ ]:
!cd ptx-puzzles && python grade.py 3

## Puzzle 4: The last block

In [ ]:
%%writefile ptx-puzzles/puzzles/04_last_block.cu
// Puzzle 4: The last block
//
// Each of 64 blocks sums its slice of the input, and thread 0 writes the block's partial sum. Thread 0 then takes a
// ticket from a counter with atomicAdd. The block that draws the last ticket calls fence_acq_rel() and adds up all 64
// partials.
//
// Make the last block read every partial. Par: 1.
//
// From: CUDA's threadFenceReduction sample and the reductions built on it.
#include "common.cuh"

constexpr int B = 64, T = 128;

__device__ unsigned block_sum(unsigned* s) {      // sums s[0..T) in place; every thread gets the total
  for (int k = T / 2; k > 0; k >>= 1) { if (threadIdx.x < k) s[threadIdx.x] += s[threadIdx.x + k]; __syncthreads(); }
  return s[0];
}

__global__ void sum_last_block(const unsigned* in, unsigned* partial, unsigned* tickets, unsigned* total) {
  __shared__ unsigned s[T];
  __shared__ bool last;
  const int t = threadIdx.x, b = blockIdx.x;
  s[t] = in[b * T + t];
  __syncthreads();
  const unsigned mine = block_sum(s);
  if (t == 0) {
    partial[b] = mine;
    // >>> edit
    last = atomicAdd(tickets, 1u) == gridDim.x - 1;
    // <<< edit
  }
  __syncthreads();
  if (!last) return;
  if (t == 0) fence_acq_rel();
  __syncthreads();
  s[t] = t < B ? partial[t] : 0;
  __syncthreads();
  const unsigned all = block_sum(s);
  if (t == 0) *total = all;
}

int main() {
  unsigned h[B * T], want = 0;
  for (int i = 0; i < B * T; ++i) { h[i] = i % 7 + 1; want += h[i]; }
  unsigned* in = device_copy(h, B * T);
  unsigned* partial = device_zeros<unsigned>(B);
  unsigned* tickets = device_zeros<unsigned>(1);
  unsigned* total = device_zeros<unsigned>(1);
  sum_last_block<<<B, T>>>(in, partial, tickets, total);
  finish_launch();
  unsigned got;
  host_copy(&got, total, 1);
  return check("total", &got, &want, 1);
}


In [ ]:
!cd ptx-puzzles && python grade.py 4

## Puzzle 5: One thread signals for the block

In [ ]:
%%writefile ptx-puzzles/puzzles/05_block_signal.cu
// Puzzle 5: One thread signals for the block
//
// Block 1's 128 threads each write one element of x. Thread 0 then calls __threadfence() and increments a
// counter. Block 0 waits on the counter with an acquire and reads x.
//
// Make every element of x reach block 0. Par: 1.
//
// From: the usual megakernel signal, where one thread bumps the counter for its whole block.
#include "common.cuh"

constexpr int T = 128;

__global__ void handoff(unsigned* ready, float* x, float* y) {
  const int t = threadIdx.x;
  if (blockIdx.x == 1) {                          // producer
    x[t] = 3.0f * t;
    // >>> edit
    // <<< edit
    if (t == 0) { __threadfence(); atomicAdd(ready, 1u); }
  } else {                                        // consumer
    if (t == 0) { while (*(volatile unsigned*)ready == 0) {} __threadfence(); }
    __syncthreads();
    y[t] = x[t] - 1.0f;
  }
}

int main() {
  unsigned* ready = device_zeros<unsigned>(1);
  float* x = device_zeros<float>(T);
  float* y = device_zeros<float>(T);
  handoff<<<2, T>>>(ready, x, y);
  finish_launch();
  float got[T], want[T];
  host_copy(got, y, T);
  for (int t = 0; t < T; ++t) want[t] = 3.0f * t - 1.0f;
  return check("y", got, want, T);
}


In [ ]:
!cd ptx-puzzles && python grade.py 5

## Puzzle 6: One thread waits for the block

In [ ]:
%%writefile ptx-puzzles/puzzles/06_block_wait.cu
// Puzzle 6: One thread waits for the block
//
// Block 1 writes x and signals correctly. In block 0, thread 0 spins on the counter and then fences. The other
// threads read x. Warps 1 to 3 first do about 2 ms of other work, and by then block 1 has long finished, so on real
// hardware their reads almost always see the new values.
//
// Make every thread in block 0 read x after block 1 wrote it. Par: 1.
//
// From: the usual megakernel wait, where one thread waits for its whole block.
#include "common.cuh"

constexpr int T = 128;

__device__ void other_work() { for (int i = 0; i < 20; ++i) __nanosleep(100000); }

__global__ void handoff(unsigned* ready, float* x, float* y) {
  const int t = threadIdx.x;
  if (blockIdx.x == 1) {                          // producer
    x[t] = 5.0f * t;
    __syncthreads();
    if (t == 0) { __threadfence(); atomicAdd(ready, 1u); }
  } else {                                        // consumer
    if (t == 0) { while (*(volatile unsigned*)ready == 0) {} __threadfence(); }
    else if (t >= 32) other_work();
    // >>> edit
    // <<< edit
    y[t] = x[t] + 2.0f;
  }
}

int main() {
  unsigned* ready = device_zeros<unsigned>(1);
  float* x = device_zeros<float>(T);
  float* y = device_zeros<float>(T);
  handoff<<<2, T>>>(ready, x, y);
  finish_launch();
  float got[T], want[T];
  host_copy(got, y, T);
  for (int t = 0; t < T; ++t) want[t] = 5.0f * t + 2.0f;
  return check("y", got, want, T);
}


In [ ]:
!cd ptx-puzzles && python grade.py 6

## Puzzle 7: A grid barrier

In [ ]:
%%writefile ptx-puzzles/puzzles/07_grid_barrier.cu
// Puzzle 7: A grid barrier
//
// A software grid barrier. At each barrier, every block's thread 0 reads the generation number, fences, and takes a
// ticket from a counter. The last block to arrive resets the counter to 0, fences, and increments the generation. The
// other blocks spin on the generation and fence. The kernel runs 1,024 steps with a barrier after each, because the bug
// only shows when two blocks arrive at almost the same moment.
//
// Make every step read the previous step's values. Par: 0.
//
// From: Lucebox's AtomicGridSync.
#include "common.cuh"

constexpr int B = 8, T = 128, N = B * T, STEPS = 1024;

__device__ void grid_sync(unsigned* count, unsigned* gen) {
  __syncthreads();
  if (threadIdx.x == 0) {
    const unsigned g = *(volatile unsigned*)gen;
    __threadfence();
    if (atomicAdd(count, 1u) == gridDim.x - 1) {
      // >>> edit
      *count = 0;
      // <<< edit
      __threadfence();
      atomicAdd(gen, 1u);
    } else {
      while (*(volatile unsigned*)gen == g) {}
      __threadfence();
    }
  }
  __syncthreads();
}

__global__ void stencil(unsigned* a, unsigned* b, unsigned* count, unsigned* gen) {
  const int i = blockIdx.x * T + threadIdx.x;
  for (int s = 0; s < STEPS; ++s) {
    b[i] = a[(i + N - 1) % N] + 2 * a[i] + a[(i + 1) % N] + s;   // neighbours at a block's edge live in another block
    grid_sync(count, gen);
    unsigned* c = a; a = b; b = c;
  }
}

int main() {
  unsigned h[N], tmp[N];
  for (int i = 0; i < N; ++i) h[i] = i * 2654435761u;
  unsigned* a = device_copy(h, N);
  unsigned* b = device_zeros<unsigned>(N);
  unsigned* count = device_zeros<unsigned>(1);
  unsigned* gen = device_zeros<unsigned>(1);
  stencil<<<B, T>>>(a, b, count, gen);
  finish_launch();
  for (int s = 0; s < STEPS; ++s) {
    for (int i = 0; i < N; ++i) tmp[i] = h[(i + N - 1) % N] + 2 * h[i] + h[(i + 1) % N] + s;
    for (int i = 0; i < N; ++i) h[i] = tmp[i];
  }
  unsigned got[N];
  host_copy(got, STEPS % 2 ? b : a, N);
  return check("a", got, h, N);
}


In [ ]:
!cd ptx-puzzles && python grade.py 7

## Puzzle 8: The TMA store

In [ ]:
%%writefile ptx-puzzles/puzzles/08_tma_store.cu
// Puzzle 8: The TMA store (sm_90 and newer)
//
// Each block's threads write a tile into shared memory, and the block syncs. Thread 0 then copies the tile to global
// memory with the TMA unit's bulk copy and waits for it to finish (bulk_store_and_wait in common.cuh).
//
// Make the copy read the finished tile. Par: 1.
//
// From: TMA store epilogues, such as CUTLASS's.
#include "common.cuh"

constexpr int B = 4, T = 128, TILE = 256;

__global__ void tile_store(float* g) {
  __shared__ alignas(128) float tile[TILE];
  const int t = threadIdx.x, b = blockIdx.x;
  for (int i = t; i < TILE; i += T) tile[i] = (float)(b * 1000 + i);
  // >>> edit
  // <<< edit
  __syncthreads();
  if (t == 0) bulk_store_and_wait(g + b * TILE, tile, TILE * sizeof(float));
}

int main() {
  cudaDeviceProp prop;
  CK(cudaGetDeviceProperties(&prop, 0));
  if (prop.major < 9) { printf("needs sm_90 or newer (this GPU is sm_%d%d)\n", prop.major, prop.minor); return 77; }
  float* g = device_zeros<float>(B * TILE);
  tile_store<<<B, T>>>(g);
  finish_launch();
  static float got[B * TILE], want[B * TILE];
  host_copy(got, g, B * TILE);
  for (int b = 0; b < B; ++b) for (int i = 0; i < TILE; ++i) want[b * TILE + i] = (float)(b * 1000 + i);
  return check("g", got, want, B * TILE);
}


In [ ]:
!cd ptx-puzzles && python grade.py 8

## Puzzle 9: Argmax, row by row

In [ ]:
%%writefile ptx-puzzles/puzzles/09_argmax.cu
// Puzzle 9: Argmax, row by row
//
// One block finds the argmax of each of 16 rows. For each row, every warp finds the best of its 32 values, lane 0 of
// each warp writes its candidate into shared memory, the block syncs, and warp 0 picks the row's answer from the 8
// candidates. The loop then moves to the next row and reuses the same shared slots.
//
// Make every row's answer come from that row. Par: 1.
//
// From: Mirage MPK's argmax (mirage-project/mirage#843).
#include "common.cuh"

constexpr int T = 256, W = T / 32, ROWS = 16;

__device__ void warp_argmax(float& v, int& i) {
  for (int o = 16; o > 0; o >>= 1) {
    const float v2 = __shfl_down_sync(0xffffffffu, v, o);
    const int i2 = __shfl_down_sync(0xffffffffu, i, o);
    if (v2 > v) { v = v2; i = i2; }
  }
}

__global__ void argmax_rows(const float* x, int* out) {
  __shared__ float best_val[W];
  __shared__ int best_idx[W];
  const int t = threadIdx.x, lane = t % 32, warp = t / 32;
  for (int r = 0; r < ROWS; ++r) {
    float v = x[r * T + t];
    int i = t;
    warp_argmax(v, i);
    if (lane == 0) { best_val[warp] = v; best_idx[warp] = i; }
    __syncthreads();
    if (warp == 0) {
      v = lane < W ? best_val[lane] : -1.0f;
      i = lane < W ? best_idx[lane] : -1;
      warp_argmax(v, i);
      if (lane == 0) out[r] = i;
    }
    // >>> edit
    // <<< edit
  }
}

int main() {
  static float h[ROWS * T];
  int want[ROWS];
  for (int r = 0; r < ROWS; ++r)
    for (int t = 0; t < T; ++t) {
      h[r * T + t] = (float)((t * 37 + r * 11) % T);   // each row is a permutation of 0..255
      if ((t * 37 + r * 11) % T == T - 1) want[r] = t;
    }
  float* x = device_copy(h, ROWS * T);
  int* out = device_zeros<int>(ROWS);
  argmax_rows<<<1, T>>>(x, out);
  finish_launch();
  int got[ROWS];
  host_copy(got, out, ROWS);
  return check("out", got, want, ROWS);
}


In [ ]:
!cd ptx-puzzles && python grade.py 9

## Puzzle 10: Recycling a page

In [ ]:
%%writefile ptx-puzzles/puzzles/10_page_recycle.cu
// Puzzle 10: Recycling a page
//
// A megakernel block runs its instructions out of a page of shared memory, and each warp has one job. For the first
// instruction, the consumer (warp 0) writes its output O into the first half of the page and arrives on barrier O_READY.
// It then writes a second result, L, into the second half. The storer (warp 1) waits on O_READY, copies O to global
// memory, and frees the page by arriving on barrier FREE. The loader (warp 2) waits on FREE and loads the next
// instruction's input into the page.
//
// Make every output correct. Par: 1.
//
// From: Hazy Research's Llama-1B megakernel (HazyResearch/Megakernels#17).
#include "common.cuh"

constexpr int H = 32;                    // a page holds O[0..H) and then L[0..H)
constexpr int O_READY = 1, FREE = 2, LOADED = 3;
// >>> edit
constexpr int FREE_THREADS = 64;         // threads that meet at FREE: the storer and the loader
// <<< edit

__global__ void megakernel_block(const float* in1, const float* in2, float* out_o, float* out_l, float* out2) {
  __shared__ float page[2 * H];
  const int lane = threadIdx.x % 32, warp = threadIdx.x / 32;
  if (warp == 0) {                       // consumer
    const float a = in1[lane];
    page[lane] = 2.0f * a;               // O
    bar_arrive(O_READY, 64);
    const float l = a + 1.0f;            // L
    page[H + lane] = l;
    out_l[lane] = l;
    // >>> edit
    // <<< edit
    bar_sync(LOADED, 64);                // second instruction
    out2[lane] = page[lane] * page[H + lane];
  } else if (warp == 1) {                // storer
    bar_sync(O_READY, 64);
    out_o[lane] = page[lane];
    bar_arrive(FREE, FREE_THREADS);
  } else {                               // loader
    bar_sync(FREE, FREE_THREADS);
    page[lane] = in2[lane];
    page[H + lane] = in2[H + lane];
    bar_arrive(LOADED, 64);
  }
}

int main() {
  float h1[H], h2[2 * H];
  for (int i = 0; i < H; ++i) h1[i] = (float)i;
  for (int i = 0; i < 2 * H; ++i) h2[i] = (float)(100 + i);
  float* in1 = device_copy(h1, H);
  float* in2 = device_copy(h2, 2 * H);
  float* out_o = device_zeros<float>(H);
  float* out_l = device_zeros<float>(H);
  float* out2 = device_zeros<float>(H);
  megakernel_block<<<1, 96>>>(in1, in2, out_o, out_l, out2);
  finish_launch();
  float got[H], want[H];
  int status = 0;
  host_copy(got, out_o, H); for (int i = 0; i < H; ++i) want[i] = 2.0f * h1[i];
  status |= check("out_o", got, want, H);
  host_copy(got, out2, H); for (int i = 0; i < H; ++i) want[i] = h2[i] * h2[H + i];
  status |= check("out2", got, want, H);
  return status;
}


In [ ]:
!cd ptx-puzzles && python grade.py 10

## Puzzle 11: Split-K, three ways

In [ ]:
%%writefile ptx-puzzles/puzzles/11_split_k.cu
// Puzzle 11: Split-K, three ways
//
// Three blocks each compute one part of the K loop for the same output tile. The first split stores its partial in a
// workspace and sets a flag with a release. The middle split waits for the flag and adds its partial into the
// workspace with atomicAdd. The final split waits for the flag, reads the workspace, adds its own partial, and writes
// the tile.
//
// Make the final split read all three partials. Par: 2.
//
// From: CUTLASS's SM90 Stream-K GEMM before 3.7.0 (NVIDIA/cutlass#2027).
#include "common.cuh"

constexpr int T = 256, SPLITS = 3, K = 12, KS = K / SPLITS;

__global__ void splitk_tile(const float* a, unsigned* flag, float* work, float* out) {
  const int s = blockIdx.x, t = threadIdx.x;
  float p = 0.0f;
  for (int k = s * KS; k < (s + 1) * KS; ++k) p += a[k * T + t];
  if (s == 0) {                                   // first split
    work[t] = p;
    __syncthreads();
    if (t == 0) { __threadfence(); atomicAdd(flag, 1u); }
  } else if (s == 1) {                            // middle split
    if (t == 0) { while (*(volatile unsigned*)flag < 1) {} __threadfence(); }
    __syncthreads();
    for (int i = 0; i < 10; ++i) __nanosleep(100000);   // its K range is longer: Stream-K splits are uneven
    atomicAdd(&work[t], p);
    // >>> edit
    // <<< edit
  } else {                                        // final split
    if (t == 0) {
      // >>> edit
      while (*(volatile unsigned*)flag < 1) {}
      // <<< edit
      __threadfence();
    }
    __syncthreads();
    out[t] = work[t] + p;
  }
}

int main() {
  static float h[K * T];
  float want[T];
  for (int t = 0; t < T; ++t) {
    want[t] = 0.0f;
    for (int k = 0; k < K; ++k) { h[k * T + t] = (float)((k * 7 + t) % 13); want[t] += h[k * T + t]; }
  }
  float* a = device_copy(h, K * T);
  unsigned* flag = device_zeros<unsigned>(1);
  float* work = device_zeros<float>(T);
  float* out = device_zeros<float>(T);
  splitk_tile<<<SPLITS, T>>>(a, flag, work, out);
  finish_launch();
  float got[T];
  host_copy(got, out, T);
  return check("out", got, want, T);
}


In [ ]:
!cd ptx-puzzles && python grade.py 11

## Puzzle 12: The spinlock

In [ ]:
%%writefile ptx-puzzles/puzzles/12_spinlock.cu
// Puzzle 12: The spinlock
//
// 64 blocks each fold one value into a shared record (a sum, a count and a maximum) under a lock. Thread 0 takes the
// lock with atomicCAS, updates the record with plain loads and stores, and frees the lock with atomicExch.
//
// Make the lock protect the record. Par: 2.
//
// From: the textbook GPU spinlock.
#include "common.cuh"

constexpr int B = 64;

struct Record { unsigned sum, count, max; };

__global__ void locked_update(unsigned* lock, Record* rec) {
  if (threadIdx.x != 0) return;
  const unsigned v = blockIdx.x * 37 % B + 1;
  while (atomicCAS(lock, 0u, 1u) != 0u) {}
  // >>> edit
  // <<< edit
  rec->sum += v;
  rec->count += 1;
  if (v > rec->max) rec->max = v;
  // >>> edit
  // <<< edit
  atomicExch(lock, 0u);
}

int main() {
  unsigned* lock = device_zeros<unsigned>(1);
  Record* rec = device_zeros<Record>(1);
  locked_update<<<B, 32>>>(lock, rec);
  finish_launch();
  Record got;
  host_copy(&got, rec, 1);
  unsigned g[3] = {got.sum, got.count, got.max}, want[3] = {0, B, 0};
  for (int b = 0; b < B; ++b) { const unsigned v = b * 37 % B + 1; want[0] += v; if (v > want[2]) want[2] = v; }
  return check("record", g, want, 3);
}


In [ ]:
!cd ptx-puzzles && python grade.py 12

## Solutions

The reference solutions, with a paragraph on why each one works, are in [solutions/](https://github.com/ShreshthRajan/ptx-puzzles/blob/main/solutions/README.md).